In [1]:
import pandas as pd

raw    = pd.read_csv('../dataset/raw/order_items.csv',
                     parse_dates=['created_at', 'shipped_at', 'delivered_at', 'returned_at'])
clean  = pd.read_parquet('../dataset/processed/clean_order_items.parquet')
orders = pd.read_csv('../dataset/raw/orders.csv', usecols=['order_id', 'created_at'])
orders['created_at'] = pd.to_datetime(orders['created_at'], utc=True, format='mixed').dt.tz_localize(None)
exc    = pd.read_parquet('../dataset/processed/exception_order_items.parquet')
assert len(raw) == len(clean) + len(exc), 'record counts do not match'

retention = pd.DataFrame({'records': [len(raw), len(clean), len(exc)]},
                         index=['raw', 'clean (kept)', 'exception (dropped)'])
retention['share_of_raw_%'] = (100 * retention['records'] / len(raw)).round(2)
retention

def kpis(df):
    returned  = (df['status'] == 'Returned').sum()
    delivered = df['status'].isin(['Complete', 'Returned']).sum()
    gmv       = df['sale_price'].sum()
    net_rev   = df.loc[~df['status'].isin(['Returned', 'Cancelled']), 'sale_price'].sum()
    n_orders  = df['order_id'].nunique()
    return pd.Series({
        'items': len(df), 'orders': n_orders, 'users': df['user_id'].nunique(),
        'GMV': gmv, 'net_revenue': net_rev, 'AOV': gmv / n_orders,
        'items_per_order': len(df) / n_orders,
        'return_rate_all_items_%': 100 * returned / len(df),
        'return_rate_delivered_%': 100 * returned / delivered,
        'return_rate_amount_%': 100 * df.loc[df['status'] == 'Returned', 'sale_price'].sum() / gmv,
    })

summary = pd.DataFrame({'raw': kpis(raw), 'clean': kpis(clean), 'dropped': kpis(exc)})
summary['change (clean - raw)'] = summary['clean'] - summary['raw']
summary['change_%'] = 100 * summary['change (clean - raw)'] / summary['raw']
summary.round(2)

mix = pd.concat({'raw': raw['status'].value_counts(),
                 'clean': clean['status'].value_counts(),
                 'dropped': exc['status'].value_counts()}, axis=1).fillna(0).astype(int)
mix['dropped_%_of_status'] = (100 * mix['dropped'] / mix['raw']).round(1)
for s in ['raw', 'clean', 'dropped']:
    mix[f'{s}_mix_%'] = (100 * mix[s] / mix[s].sum()).round(1)
mix

exc_orders, clean_orders = set(exc['order_id']), set(clean['order_id'])
pd.Series({'orders_with_dropped_items': len(exc_orders),
           'fully_removed': len(exc_orders - clean_orders),
           'partially_removed (some items kept)': len(exc_orders & clean_orders)})

o = orders.rename(columns={'created_at': 'order_created_at'})
def by_quarter(df, name):
    d = df.merge(o, on='order_id', how='left')
    return d.groupby(d['order_created_at'].dt.to_period('Q'))['sale_price'].sum().rename(name)

by_q = pd.concat([by_quarter(raw, 'raw_GMV'), by_quarter(clean, 'clean_GMV')], axis=1)
by_q['gmv_lost_%'] = (100 * (1 - by_q['clean_GMV'] / by_q['raw_GMV'])).round(1)
by_q

,raw_GMV,clean_GMV,gmv_lost_%
order_created_at,,,
2019Q1,4.298790e+03,3.540340e+03,17.6
2019Q2,1.628682e+04,1.309960e+04,19.6
2019Q3,2.858443e+04,2.292127e+04,19.8
2019Q4,3.771285e+04,3.001745e+04,20.4
2020Q1,5.199879e+04,4.192404e+04,19.4
2020Q2,6.511917e+04,5.063411e+04,22.2
2020Q3,7.564495e+04,5.961973e+04,21.2
2020Q4,9.356308e+04,7.578775e+04,19.0
2021Q1,1.002705e+05,8.132771e+04,18.9


In [2]:
mart = pd.read_parquet('../dataset/processed/mart_financial_summary.parquet')

chk = by_q.copy()
chk.index = [f"{p.year}-Q{p.quarter}" for p in chk.index]
cmp = mart.set_index('year_quarter')[['GMV', 'total_items_sold', 'return_rate_pct']].join(chk['clean_GMV'])
cmp['mart / notebook'] = (cmp['GMV'] / cmp['clean_GMV']).round(3)
cmp.tail(9)

,GMV,total_items_sold,return_rate_pct,clean_GMV,mart / notebook
year_quarter,,,,,
2024-Q4,392380.00,6522,9.14,3.923800e+05,1.0
2025-Q1,417073.95,6997,8.85,4.170740e+05,1.0
2025-Q2,464941.83,7855,9.18,4.649418e+05,1.0
2025-Q3,507144.41,8517,8.17,5.071444e+05,1.0
2025-Q4,599326.40,9986,8.89,5.993264e+05,1.0
2026-Q1,653836.11,11087,8.91,6.538361e+05,1.0
2026-Q2,826775.51,13847,8.79,8.267755e+05,1.0
2026-Q3,1141574.81,19062,8.91,1.141575e+06,1.0
2026-Q4,381307.74,6402,8.79,3.813077e+05,1.0


In [3]:
exc_orders, clean_orders = set(exc['order_id']), set(clean['order_id'])
order_impact = pd.Series({
    'orders_with_dropped_items': len(exc_orders),
    'fully_removed': len(exc_orders - clean_orders),
    'partially_removed (some items kept)': len(exc_orders & clean_orders),
})

display(summary.round(2))
display(mix)
display(order_impact)

,raw,clean,dropped,change (clean - raw),change_%
items,181466.00,145577.00,35889.00,-35889.00,-19.78
orders,124978.00,117126.00,20557.00,-7852.00,-6.28
users,79987.00,76767.00,18885.00,-3220.00,-4.03
GMV,10779102.74,8657254.17,2121848.57,-2121848.57,-19.68
net_revenue,8081343.59,6294871.01,1786472.58,-1786472.58,-22.11
AOV,86.25,73.91,103.22,-12.33,-14.30
items_per_order,1.45,1.24,1.75,-0.21,-14.40
return_rate_all_items_%,10.10,8.77,15.53,-1.34,-13.22
return_rate_delivered_%,28.76,28.71,28.86,-0.04,-0.15
return_rate_amount_%,10.13,8.74,15.81,-1.39,-13.73


,raw,clean,dropped,dropped_%_of_status,raw_mix_%,clean_mix_%,dropped_mix_%
status,,,,,,,
Shipped,54263,37684,16579,30.6,29.9,25.9,46.2
Complete,45430,31692,13738,30.2,25.0,21.8,38.3
Processing,36469,36469,0,0.0,20.1,25.1,0.0
Cancelled,26967,26967,0,0.0,14.9,18.5,0.0
Returned,18337,12765,5572,30.4,10.1,8.8,15.5


orders_with_dropped_items              20557
fully_removed                           7852
partially_removed (some items kept)    12705
dtype: int64

**Timestamp cleaning removes 19.8% of order items** 
- 35,889 of 181,466 order items (145,577 kept) are dropped because a shipped, delivered or returned timestamp is earlier than the item's `created_at`, which would give negative fulfilment times. The rule only hits items that have shipped: about 30% of Shipped, Complete and Returned items are dropped, and no Processing or Cancelled items are, so Processing and Cancelled items rise from 35% to 44% of the cleaned data. Compared with the raw data, the cleaned data understates GMV by 19.7% and net revenue by 22.1%, and the all-items return rate falls from 10.1% to 8.8%. The return rate among delivered items is essentially unchanged (28.8% raw vs 28.7% cleaned) because Complete and Returned items are dropped at almost the same rate (30.2% vs 30.4%). 20,557 orders (16.4%) lose at least one item: 7,852 are removed entirely and 12,705 keep only some of their items, so AOV (86.25 raw vs 73.91 cleaned) and items per order (1.45 vs 1.24) are each understated by about 14%.